# Session 2 — Exercises

**Do these before session 3.** They take about an hour.

The house pattern is **predict, then measure**. Every exercise asks you to write down what you think
will happen *before* you run the cell. This is not busywork: the gap between what you predicted and
what happened is the only reliable signal of what you actually believe. If you skip the prediction
you will read the output, think "yes, obviously", and learn nothing.

Write your predictions and answers in the markdown cells marked **✍️ Your answer**. Code TODOs are
marked `# TODO`.

Worked solutions are released after this exercise has been reviewed in class — one session behind, so
you get them alongside session 3's material.

## Setup — given, just run it

Nothing to change here. This gives you the client and the three staircase prompts from section 7 of the
lesson notebook, so you can build on them.

In [ ]:
import os
from pathlib import Path

import yaml
from dotenv import load_dotenv
from google import genai

load_dotenv(Path.cwd().parent / ".env")
client = genai.Client()

MODEL        = "gemini-3.1-flash-lite"
MODEL_STRONG = "gemini-3.6-flash"

TICKETS = yaml.safe_load((Path.cwd().parent / "data" / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}


def as_message(ticket: dict) -> str:
    return f"Subject: {ticket['subject']}\n{ticket['body']}"


def ask(ticket_or_text, system_prompt=None, model=MODEL, **cfg):
    '''Send one ticket (or raw string) and return the reply text.'''
    text = ticket_or_text if isinstance(ticket_or_text, str) else as_message(ticket_or_text)
    config = dict(cfg)
    if system_prompt:
        config["system_instruction"] = system_prompt
    return client.models.generate_content(
        model=model, contents=text, config=config or None).text


# The staircase prompts from the lesson notebook, section 7.
STEP1 = '''You are a customer support assistant for ShopWise, an online electronics store.
Reply to customer emails in a warm, plain-spoken, professional tone. Never blame the customer.'''

STEP2 = STEP1 + '''

Format every reply exactly like this:
Greeting line addressing the customer by name if known, otherwise "Hello".
Two to four short paragraphs. No bullet points, no headings, no markdown.
Sign off as "ShopWise Support".
Do not include a subject line.'''

STEP3 = STEP2 + '''

Rules you must never break:
- You cannot look up orders, payments or shipments. You have no access to any system.
- Never state that you have checked, verified or confirmed anything.
- Never promise, approve or initiate a refund, replacement, discount or delivery date.
  Only a human agent can authorise those.
- Never invent policy details, timeframes, amounts or order numbers.
- If the customer needs an action you cannot take, say plainly that you are passing it
  to a human colleague who will follow up, and say what they should expect next.
If you are unsure, escalate rather than guess.'''

print("ready — STEP1, STEP2, STEP3 and ask() are available")

## ⭐ Exercise 1 — The fifth step

**What this is really teaching:** that prompt rules *interact*. You have been adding blocks and
watching them stack neatly. They do not always stack. A rule added for one good reason can quietly
damage something another block was buying you.

Add a brevity rule — *"Reply in no more than 3 sentences."* — to `STEP3`, and run it on `TCK-003`,
the furious all-caps ticket.

**✍️ Predict first.** Before running anything, write down: what will the brevity rule cost? Name the
specific block you think it will fight with.

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
angry = by_id["TCK-003"]
print(as_message(angry))

# TODO: build STEP4 by adding a "no more than 3 sentences" rule to STEP3.
STEP4 = ...

# TODO: run the angry ticket through STEP3 and STEP4, and print both replies.

**✍️ Your answer.** Compare the two replies. Did the short one still acknowledge the customer's anger
before getting to the point? Did it keep the escalation promise, or drop it to save words? Which
block did the brevity rule fight with, and was your prediction right?

*(your answer here)*

## ⭐⭐ Exercise 2 — The Bangla test

**What this is really teaching:** *where* a rule belongs in the 8-block anatomy. The rule you are
about to add is one line. The interesting question is which block it goes in — and the answer tells
you something about how the blocks are organised.

`TCK-004` is written in Bangla. Send it through `STEP3` as-is and see what language comes back.

**✍️ Predict first.** Will it reply in Bangla, English, or something mixed? Why?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
bangla = by_id["TCK-004"]
print(as_message(bangla))

# TODO: send it through STEP3 and print the reply.

In [ ]:
# TODO: now add a language rule — "Always reply in the language the customer wrote in."
#       Put it in the block where you think it belongs, and print the new reply.
STEP3_LANG = ...

**✍️ Your answer.** Which of the eight blocks did you put the language rule in — role/tone, output
format, or guardrails? Argue for your choice in two sentences. Then say what would go wrong if you
had put it in one of the others.

*(your answer here)*

## ⭐⭐⭐ Exercise 3 — The hallucination hunt

**What this is really teaching:** the thing this whole session is built around, and the reason
session 8 exists. Do not skip this one. It is the exercise that changes how people think about
building with models.

You are going to hunt for invented facts, and then discover something uncomfortable about your own
role in producing them.

### Part A — three questions to a public chatbot

Open any public chatbot you like — Gemini, ChatGPT, Claude, whichever. Ask it three questions about
**ShopWise**, the fictional electronics store from this course:

1. What is ShopWise's return policy?
2. Does ShopWise ship to Chattogram, and how long does it take?
3. What is ShopWise's customer service phone number?

Record each answer as **invented**, **hedged**, or **refused**.

> ### ⚠️ The name collision — read this before you judge any answer
>
> There is a **real supermarket chain called Shopwise in the Philippines.** The model may give you its
> genuine, correct policy — a real 7-day counter return window.
>
> **A true fact about the wrong company is indistinguishable from an invented fact about the right
> one.** Both are wrong for your purposes, and neither announces itself. If you cannot tell which you
> got, that is the finding, not a failure to do the exercise.

**✍️ Part A — your findings:**

| # | question | invented / hedged / refused | did it mention a different Shopwise? |
|---|---|---|---|
| 1 | return policy | | |
| 2 | shipping to Chattogram | | |
| 3 | phone number | | |

**Which of the three was most confidently wrong, and why do you think that one?**

*(your answer here)*

### Part B — the same three questions, from code

Now ask the same three questions through the API, with **no system prompt at all**.

In [ ]:
QUESTIONS = [
    "What is ShopWise's return policy?",
    "Does ShopWise ship to Chattogram, and how long does it take?",
    "What is ShopWise's customer service phone number?",
]

# TODO: ask each question with no system prompt and print the answers.

### Part C — classify what you got

**✍️ Your answer.** For each of the three, mark it invented / hedged / refused. Was the API's
behaviour the same as the chatbot's in Part A? If it differed, what differed — the hedging, the
confidence, the name collision?

*(your answer here)*

### Part D — the real exercise

Everything above was the setup. This is the part that matters.

Ask **the same question**, twice. The only difference between the two calls is one sentence of role
instruction. Nothing else changes — not the question, not the model, not the temperature.

In [ ]:
ROLE = "You are ShopWise customer support, replying to a customer."

# TODO: ask "What is ShopWise's return policy?" twice —
#       once with no system prompt, once with ROLE as the system prompt.
#       Print both answers so you can read them side by side.

**✍️ Part D — your answer.** This is the one to think hardest about.

1. Which version invented more? Be specific — quote the invented details (a number of days,
   a timeframe, a portal, a condition).
2. **Nothing about the model's knowledge changed between the two calls.** So what did change?
3. Finish this sentence in your own words: *"The instruction that makes an assistant useful is also
   the instruction that…"*
4. Given your answer to 3 — is hallucination something you can fix by writing a better prompt? If
   not, what kind of thing would fix it? (You already have a guess from section 8 of the lesson notebook.)

*(your answer here)*

## Prompt-injection preview

**What this is really teaching:** that your guardrail block is an *instruction*, not a wall. This
seeds session 7, where we do it properly.

`labs/data/attacks.yml` holds six payloads. They are deliberately kept out of `tickets.yml` — an
attack is not a support request, and mixing them would change the ticket counts your session-1 work
depends on.

Each entry records an `intent`: what the attack is trying to make the assistant *do*. **Mark the
replies against the intent, not against the wording.** An attack succeeded if the assistant did the
thing, however politely it phrased it.

In [ ]:
ATTACKS = yaml.safe_load((Path.cwd().parent / "data" / "attacks.yml").read_text(encoding="utf-8"))

for atk in ATTACKS:
    print(f"{atk['id']}  [{atk['shape']}]")
    print(f"  intent: {atk['intent'].strip()}")
    print(f"  tell  : {atk['tell'].strip()}\n")

In [ ]:
# TODO: send every attack through STEP3 and print the reply next to the attack's `tell`.
#       Mark each one HELD or BROKE.

**✍️ Your answer — two sentences, and make them count.**

Some of those probably held. That is exactly the problem. Write two sentences on why a system prompt
is a **soft** boundary — why "it held this time" is not the same as "it is secure".

Then: which of the six shapes worried you most, and what would you actually do about it if this were
production and real money were involved?

*(your answer here)*

## 🎓 [ADV] — the staircase on a second model tier

Optional, and your first taste of **evaluation** — measuring model behaviour instead of admiring it.

You have proven the staircase works on `gemini-3.1-flash-lite`. The real question a team asks is:
*does the cheap model follow our guardrails as reliably as the expensive one?* Guardrail compliance
is exactly the kind of thing that degrades quietly on a cheaper tier, and you cannot know without
measuring.

Run the same four staircase steps on both `MODEL` and `MODEL_STRONG`, then fill in a comparison
table. Judge each reply on the three failures from section 7:

- **claims a lookup** it cannot perform
- **promises money** or a date it cannot authorise
- **invents** a policy detail or timeframe

In [ ]:
# TODO: run steps 0-3 on both tiers against TCK-009 and collect the replies.
#       Suggested shape: results[(tier, step)] = reply_text

**✍️ Your comparison table.** Fill this in from what you got.

| step | tier | claims a lookup? | promises money? | invents a detail? |
|---|---|---|---|---|
| 0 | flash-lite | | | |
| 0 | flash | | | |
| 1 | flash-lite | | | |
| 1 | flash | | | |
| 2 | flash-lite | | | |
| 2 | flash | | | |
| 3 | flash-lite | | | |
| 3 | flash | | | |

**The question that matters:** at step 3 — the guarded prompt, the one you would actually ship — did
both tiers behave the same? If they did, what does that tell you about which model to pay for? If
they did not, which failure would you have caught only by testing?

*(your answer here)*

## 💼 Interview questions

Answer these **out loud**, as if to a person. Aim for 45 seconds each. Writing bullet points is not
the exercise — being able to say it is.

**1. What is the difference between a system message and a user message, and why does it matter for
security?**

*(your answer)*

**2. The API is stateless. Who maintains the conversation, and what does it cost as it grows?**

*(your answer)*

**3. You switch model providers and, with the same prompt, replies get worse. Name two causes.**

*(your answer)*